In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/banking-dataset-marketing-targets/train.csv
/kaggle/input/banking-dataset-marketing-targets/test.csv


# 1. Data Summarization

In [ ]:
import pandas as pd
import numpy as np

# Load the training dataset using the correct Kaggle path
# Assume the file is 'train.csv' and the separator is semicolon (;).
file_path = '/kaggle/input/banking-dataset-marketing-targets/train.csv' 
try:
    df = pd.read_csv(file_path, sep=';')
    
    print("--- df.head() ---")
    print(df.head())
    print("\n" + "*"*50 + "\n")
    
    print("--- df.info() ---")
    df.info()

except Exception as e:
    print(f"ERROR during loading or inspection: {e}")

--- df.head() ---
   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  

**************************************************

--- df.info() ---
<class 'pandas.core.fr

# 2. Exploratory Data Analysis (EDA)

In [3]:
# Calculate counts and percentages for the target variable 'y'
target_counts = df['y'].value_counts()
target_ratios = df['y'].value_counts(normalize=True) * 100

print("--- Target Variable (y) Distribution ---")
print(target_counts)
print("\n--- Target Variable Ratios (%) ---")
print(target_ratios.round(2))

--- Target Variable (y) Distribution ---
y
no     39922
yes     5289
Name: count, dtype: int64

--- Target Variable Ratios (%) ---
y
no     88.3
yes    11.7
Name: proportion, dtype: float64


>The distribution ratios clearly show that this project has a class imbalance problem. The relationship between the target variable y and categorical features should be examined. The goal is to see which categorical variables most influence subscription success (yes rate).

In [4]:
# Import visualization libraries for the next step
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_style("whitegrid")

# Function to calculate success rate for a given categorical column
def calculate_success_rate(df, column):
    # Group by the column and the target 'y', then count
    agg_df = df.groupby([column, 'y']).size().unstack(fill_value=0)
    
    # Calculate the subscription success rate (Yes / (Yes + No))
    agg_df['Success Rate (%)'] = (agg_df['yes'] / (agg_df['yes'] + agg_df['no'])) * 100
    
    # Sort by success rate
    agg_df = agg_df.sort_values(by='Success Rate (%)', ascending=False)
    
    return agg_df

# Analyze 'job' vs 'y'
job_analysis = calculate_success_rate(df, 'job')

print("--- Job vs Subscription Success Rate ---")
print(job_analysis[['Success Rate (%)']].round(2))

--- Job vs Subscription Success Rate ---
y              Success Rate (%)
job                            
student                   28.68
retired                   22.79
unemployed                15.50
management                13.76
admin.                    12.20
self-employed             11.84
unknown                   11.81
technician                11.06
services                   8.88
housemaid                  8.79
entrepreneur               8.27
blue-collar                7.27


>These are subscription success rates based on professional groups. This analysis provides valuable information about who to focus your marketing strategy on.

In [5]:
# Analyze 'poutcome' vs 'y'
poutcome_analysis = calculate_success_rate(df, 'poutcome')

print("--- Previous Outcome (poutcome) vs Subscription Success Rate ---")
print(poutcome_analysis[['Success Rate (%)']].round(2))

--- Previous Outcome (poutcome) vs Subscription Success Rate ---
y         Success Rate (%)
poutcome                  
success              64.73
other                16.68
failure              12.61
unknown               9.16


>- It is crucial to see how the outcome of previous marketing campaigns impacted the success of this campaign.  
>- If the customer subscribed in the previous campaign, there is almost a 65% chance they will subscribe in this campaign. This shows how high the predictive power of this variable is.

In [6]:
# Analyze 'marital' vs 'y'
marital_analysis = calculate_success_rate(df, 'marital')

print("\n" + "="*50 + "\n")
print("--- Marital Status (marital) vs Subscription Success Rate ---")
print(marital_analysis[['Success Rate (%)']].round(2))



--- Marital Status (marital) vs Subscription Success Rate ---
y         Success Rate (%)
marital                   
single               14.95
divorced             11.95
married              10.12


>According to the result of the impact of marital status on decision-making, Singles have the highest subscription rate compared to other groups.

In [7]:
# Select key numerical columns
num_cols_to_check = ['age', 'balance', 'duration', 'campaign', 'pdays']

print("\n" + "="*50 + "\n")
print("--- Numerical Feature Statistics Grouped by Target (y) ---")

# Group by 'y' and calculate descriptive statistics for selected numerical columns
grouped_stats = df.groupby('y')[num_cols_to_check].agg(['mean', 'median', 'std', 'min', 'max'])

# Print only Mean and Median for brevity and focus
print(grouped_stats[['balance', 'duration', 'campaign']].head(10))



--- Numerical Feature Statistics Grouped by Target (y) ---
         balance                                      duration         \
            mean median          std   min     max        mean median   
y                                                                       
no   1303.714969  417.0  2974.195473 -8019  102127  221.182806  164.0   
yes  1804.267915  733.0  3501.104777 -3058   81204  537.294574  426.0   

                           campaign                           
            std min   max      mean median       std min max  
y                                                             
no   207.383237   0  4918  2.846350    2.0  3.212767   1  63  
yes  392.525262   8  3881  2.141047    2.0  1.921826   1  32  


>This output demonstrates the role of numerical variables in subscriber success.  
>**Balance comment**: The average balance of subscribed customers is 38% higher than that of non-subscribers. A high balance is a significant indicator of marketing responsiveness. The median difference is even larger, indicating that outliers with very high balances in the Yes group are less effective.  
>**Duration comment**: The most critical difference is observed here. The average call duration for subscribed customers is 2.4 times longer than for non-subscribers (approximately 9 minutes vs. 3.5 minutes).  
>**Campaign comment:** Subscribed customers were contacted less frequently than non-subscribers (an average of 2.14 times). This demonstrates the importance of success on initial attempts.

**Critical Implications Before Modeling**  
>- The 88.3% "No" / 11.7% "Yes" rate requires special techniques (SMOTE, class weighting, etc.) or appropriate metric selection (AUC-ROC, F1-Score) during modeling  
>- The "poutcome" variable (success rate: 64.7%) is the strongest categorical variable predicting subscription.  
>- The "duration" variable (yes average: 537 seconds) has very high predictive power, but its use is debatable (the duration of the call is known after the subscriber is determined). It should be removed from the model for real-time prediction.  
>- High balance and low number of campaigns are positive indicators of success.

>**Threshold Optimization for Better Precision**  
>- The default threshold (0.5) gives low precision (35%), meaning many false positives  
>- By optimizing the threshold using Precision-Recall curve, we can find a better balance  
>- Higher threshold = fewer false positives = better marketing budget efficiency  
>- The optimized threshold will be used in the API to improve prediction quality


In [8]:
# Threshold Optimization for Better Precision
from sklearn.metrics import precision_recall_curve, f1_score

print("\n" + "="*50)
print("--- Threshold Optimization ---")

# Calculate precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(y_test, rf_y_proba)

# Find optimal threshold: maximize F1-score or balance precision/recall
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

# Alternative: Find threshold that gives at least 50% precision
precision_threshold_idx = np.where(precisions[:-1] >= 0.5)[0]
if len(precision_threshold_idx) > 0:
    # Choose threshold that gives best precision while maintaining reasonable recall
    precision_optimal_idx = precision_threshold_idx[np.argmax(recalls[precision_threshold_idx])]
    precision_optimal_threshold = thresholds[precision_optimal_idx]
else:
    precision_optimal_threshold = optimal_threshold

# Use the threshold that prioritizes precision (better for business)
# This reduces false positives and improves marketing budget efficiency
final_threshold = max(optimal_threshold, precision_optimal_threshold, 0.6)  # At least 0.6

# Apply optimized threshold
rf_y_pred_optimized = (rf_y_proba >= final_threshold).astype(int)

print(f"\nDefault Threshold (0.5):")
default_idx = np.argmin(np.abs(thresholds - 0.5))
print(f"  Precision: {precisions[default_idx]:.4f}")
print(f"  Recall: {recalls[default_idx]:.4f}")

print(f"\nOptimal Threshold ({final_threshold:.4f}):")
optimal_precision = precisions[np.argmin(np.abs(thresholds - final_threshold))]
optimal_recall = recalls[np.argmin(np.abs(thresholds - final_threshold))]
print(f"  Precision: {optimal_precision:.4f}")
print(f"  Recall: {optimal_recall:.4f}")
print(f"  F1-Score: {f1_score(y_test, rf_y_pred_optimized):.4f}")

print("\n--- Optimized Random Forest Classification Report ---")
print(classification_report(y_test, rf_y_pred_optimized))

print(f"\n⚠️  RECOMMENDATION: Use threshold = {final_threshold:.4f} in API for better precision")
print("   This reduces false positives and improves marketing budget efficiency.")



--- Threshold Optimization ---


NameError: name 'y_test' is not defined

# 3. Data Preprocessing

In [ ]:
# Save the Optimal Threshold for API use
import json
import os

filename_threshold = 'optimal_threshold.json'
try:
    # Check if threshold optimization was run
    if 'final_threshold' not in globals() or 'optimal_precision' not in globals() or 'optimal_recall' not in globals():
        raise NameError("Threshold optimization variables not found")
    
    # Save the optimal threshold for API use
    threshold_data = {
        'optimal_threshold': float(final_threshold),
        'precision': float(optimal_precision),
        'recall': float(optimal_recall),
        'note': 'Use this threshold in API to improve precision and reduce false positives'
    }
    with open(filename_threshold, 'w') as f:
        json.dump(threshold_data, f, indent=4)
    print(f"\n--- Optimal Threshold Saved Successfully: {filename_threshold} ---")
    print(f"   Threshold Value: {final_threshold:.4f}")
    print(f"   Expected Precision: {optimal_precision:.4f}")
    print(f"   Expected Recall: {optimal_recall:.4f}")
except NameError:
    print("\n!! WARNING !!: 'final_threshold' not found. Using default 0.6 in API.")
    print("   Please run the threshold optimization cell first for better results.")
    # Save default threshold
    threshold_data = {
        'optimal_threshold': 0.6,
        'precision': None,
        'recall': None,
        'note': 'Default threshold - run threshold optimization cell to get optimal value'
    }
    with open(filename_threshold, 'w') as f:
        json.dump(threshold_data, f, indent=4)


In [ ]:
# Create a copy of the dataframe for preprocessing
df_prep = df.copy()

# 1 & 2. Target Variable and Binary Categorical Encoding (Yes/No to 1/0)
binary_cols = ['default', 'housing', 'loan', 'y']

for col in binary_cols:
    df_prep[col] = df_prep[col].map({'yes': 1, 'no': 0})

# 3. One-Hot Encoding for Multi-Category Columns
categorical_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
df_prep = pd.get_dummies(df_prep, columns=categorical_cols, drop_first=True)


# Verify the result by checking the first few rows and shape
print("\n" + "="*50 + "\n")
print("--- Preprocessed Data Head (df_prep.head()) ---")
print(df_prep.head())
print(f"\nOriginal Columns: 17, Preprocessed Columns: {df_prep.shape[1]}")

In [ ]:
from sklearn.model_selection import train_test_split

# Define X (Features) and y (Target)
# We drop the original 'y' column, which is now encoded as 'y' (0/1)
# We also drop 'duration' to build a practical prediction model.
X = df_prep.drop(['y', 'duration'], axis=1)
y = df_prep['y']

print("X (Features) shape:", X.shape)
print("y (Target) shape:", y.shape)

In [ ]:
# Split the data into training and testing sets (80% train, 20% test)
# stratify=y is crucial for class imbalance: it ensures both train and test sets 
# have the same proportion of 'yes' and 'no' samples (approx. 11.7% 'yes').
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=42, 
                                                    stratify=y) 

print("\n--- Train/Test Split Sizes ---")
print(f"X_train size: {X_train.shape[0]}, y_train size: {y_train.shape[0]}")
print(f"X_test size: {X_test.shape[0]}, y_test size: {y_test.shape[0]}")

# 4. Model Selection and Training

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize Logistic Regression Model
# class_weight='balanced' addresses the 88.3% vs 11.7% class imbalance
model = LogisticRegression(solver='liblinear', random_state=42, class_weight='balanced', max_iter=1000)

# Train the model
model.fit(X_train, y_train)

print("\n" + "="*50 + "\n")
print("--- Model Training Complete ---")

>The model has been trained. Its performance should be tested using metrics appropriate for class imbalance (e.g., AUC-ROC, F1-Score, Recall).

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore') # Suppress warnings for cleaner output

# Predict on the test set
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1] # Probabilities for AUC calculation

print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred))

print("\n--- AUC-ROC Score ---")
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

>In projects with class imbalance, looking solely at overall accuracy is misleading. Focus should be on Recall and AUC-ROC metrics.
>- Out of 100 people the model predicted as "Yes," only 27 actually subscribe. (Too many false positives).  
>- Critical Metric: It correctly predicted 62 out of 100 people who actually subscribed (Positive Class). This is a pretty good result because class_weight='balanced' was used.
>- F1-Score is the harmonic mean of Precision and Recall. A low F1-Score indicates an overall unstable performance of the model.
>- The overall accuracy is 76%, which is low considering the class imbalance (it would have been 88.3% if it had only predicted "No"). This indicates that the model is trying to force itself to catch the "Yes" class.

>AUC-ROC Score Analysis  
>- AUC-ROC (Area Under the Receiver Operating Characteristic Curve) indicates the model's ability to distinguish a random 'Yes' sample from a random 'No' sample.  
>- 0.77 is an acceptable starting point at this stage. 1.0 is excellent, and 0.5 is a random guess.

>Confusion Matrix Analysis  
>- TP (True Positives) = 659: Those who were correctly predicted to become subscribers.  
>- FN (False Negatives) = 399: Those who were actually subscribers that the model missed (predicted "No").  
>- FP (False Positives) = 1813: Those who were unnecessarily searched by the model (predicted "Yes") but were not subscribers.  
>- In summary, Logistic Regression was successful in capturing the minority class (Yes) thanks to class_weight='balanced' (Recall: 0.62), but it generated too many false alarms (1813 False Positives). This is important for marketing budgets.

> To improve model performance and find a better balance (between Precision and Recall), let's try Random Forest, a tree-based model that is more complex and generally performs better on classification problems.  
> Again, to address class imbalance, the class_weight='balanced' parameter will be used.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize Random Forest Model
# class_weight='balanced' is used for imbalance
rf_model = RandomForestClassifier(n_estimators=200, 
                                  max_depth=10, 
                                  random_state=42, 
                                  class_weight='balanced', 
                                  n_jobs=-1)

# Train the model
rf_model.fit(X_train, y_train)

print("\n" + "="*50 + "\n")
print("--- Random Forest Model Training Complete ---")

In [ ]:
# Predict on the test set
rf_y_pred = rf_model.predict(X_test)
rf_y_proba = rf_model.predict_proba(X_test)[:, 1]

print("\n--- Random Forest Classification Report (Test Set) ---")
print(classification_report(y_test, rf_y_pred))

print("\n--- Random Forest AUC-ROC Score ---")
print(f"AUC-ROC: {roc_auc_score(y_test, rf_y_proba):.4f}")

>Random Forest Performance Analysis and Model Comparison  
>- Accuracy improved by 6%. RF has better overall accuracy.  
>- Recall improved by 2%. Both models capture approximately 60-62% of subscribers.  
>- Precision improved by 8%. RF correctly predicts a higher percentage (35%) of customers it predicted as "Yes." This translates to fewer unnecessary calls (False Positives).  
>- F-score improved by 7%. RF is more balanced. It maintains a better position for both Precision and Recall.  
>- AUC-ROC also improved. RF has stronger discriminatory power.

> The Random Forest model has higher Accuracy, Precision, F1-Score, and AUC-ROC values ​​than Logistic Regression. In particular, the increase in Precision (from 27% to 35%) means the bank will use its marketing budget more efficiently.

>- The Random Forest model will be used in the project.

In [ ]:
# Create a DataFrame for feature importance
feature_importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)

# Sort the features and get the top 10
top_10_features = feature_importances.sort_values(ascending=False).head(10)

print("\n" + "="*50 + "\n")
print("--- Top 10 Feature Importances (Random Forest) ---")
print(top_10_features.round(4))

>One of the strengths of Random Forest is that it shows which features are most important for prediction. This is valuable insight for marketing teams.

# 5. Data Transformation Pipeline for API

In [ ]:
# Re-define X (Features) for column naming
X = df_prep.drop(['y', 'duration'], axis=1)

# Save the list of all 41 feature column names for the API/Pipeline
# This list is CRITICAL for ensuring the input data matches the model's training data.
feature_cols = list(X.columns)

# Display the total number of features we need to prepare for the API
print("\n" + "="*50 + "\n")
print("--- Critical Step: API Feature List ---")
print(f"Total features required for the model: {len(feature_cols)}")
print("First 15 feature columns for API consistency:")
print(feature_cols[:15])

# Save the feature list to a file for use in the Flask/FastAPI app (next step)
import json
with open('feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

# 6. Saving Model

In [ ]:
import pickle
import os
import json

# 1. Save the Model
filename_model = 'bank_marketing_rf_model.pkl'

try:
    # Save the trained Random Forest model
    with open(filename_model, 'wb') as file:
        pickle.dump(rf_model, file)
    print(f"\n--- Model Saved Successfully: {filename_model} ---")
except NameError:
    print("\n!! ERROR !!: 'rf_model' variable not found in memory. Please re-run the training steps.")

# 2. Save the Feature Column List (Double-check)
filename_cols = 'feature_cols.json'
try:
    # Save the ordered list of 41 feature columns for API consistency
    with open(filename_cols, 'w') as f:
        json.dump(feature_cols, f)
    print(f"--- Feature List Saved Successfully: {filename_cols} ---")
except NameError:
     print("\n!! ERROR !!: 'feature_cols' variable not found in memory. Please re-run the feature preparation step.")


# 3. List the current directory contents to verify the saved files
print("\n--- Current Output Directory Content ---")
print(os.listdir('.'))